# Fine-tuning ProteinMPNN in POLAR directions — alkaliphile vs acidophile (neutralophile control)
One worked example of steering the *same* property (secreted-surface charge) in **opposite** directions by
swapping only the training cohort (`--cohort alkaline | acid`). Each cohort trains a case model **and** a
neutralophile-trained **control** (FT_neu). Expected: alkaliphile -> designs **acidic** (surface_net down);
acidophile -> designs **basic** (surface_net up); the FT_neu control shifts the opposite way each time.
**T4 GPU.** Upload `alkaline_polar_bundle.zip` (both cohorts' data). Saves to Drive.


## Setup


In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout.strip() or "NO GPU")
assert torch.cuda.is_available(), "Runtime > T4 GPU." 


In [ ]:
!pip -q install biotite biopython
print('deps ready')


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, shutil
DRIVE="/content/drive/MyDrive/polar_finetuning"
for d in ["outputs/evaluation","figures"]: os.makedirs(f"{DRIVE}/{d}",exist_ok=True)
for local,target in [("/content/outputs",DRIVE+"/outputs"),("/content/figures",DRIVE+"/figures")]:
    if not os.path.islink(local):
        if os.path.exists(local): shutil.rmtree(local)
        os.symlink(target,local)
print("outputs+figures -> Drive")


In [ ]:
import os, subprocess
if not os.path.exists("/content/ProteinMPNN"):
    subprocess.run(["git","clone","--depth","1","https://github.com/dauparas/ProteinMPNN.git","/content/ProteinMPNN"],check=True)
print("ready")


In [ ]:
# upload alkaline_polar_bundle.zip (train.py + alkmpnn/ + both cohorts' data:
#   {label}_parsed_*.jsonl, structures_{label}/, {label}_*_stageD.csv, {label}_natural_gap_test.csv)
import zipfile, os
from google.colab import files
up=files.upload()
for n in up:
    if n.endswith('.zip'):
        with zipfile.ZipFile(n) as z: z.extractall('/content'); print('extracted',n)
os.makedirs('/content/outputs/evaluation',exist_ok=True)


## Alkaliphile — fine-tune (expect designs ACIDIC (surface_net down))
Train the case model + the neutralophile control (FT_neu), select on val (max steer within the −3pp guardrail), eval on held-out neutralophile backbones. The `FT_neu` line in the verdict is the control.


In [ ]:
# alkaliphile: train FT_case + FT_neu (Noam warmup, checkpoint every 2 epochs)
!python /content/train.py --mpnn /content/ProteinMPNN --cohort alkaline --run alkaliphile_v1     --role case    --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 2 --schedule noam
!python /content/train.py --mpnn /content/ProteinMPNN --cohort alkaline --run alkaliphile_neu_v1 --role control --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 2 --schedule noam


In [ ]:
# alkaliphile: select checkpoint on VALIDATION (max steer within the recovery guardrail)
!python /content/alkmpnn/sweeps.py --cohort alkaline --mode checkpoint --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ckpt_dir /content/outputs/alkaliphile_v1/model_weights --split val --select maxsteer \
    --select_out /content/outputs/evaluation/alkaliphile_epoch.txt --save_csv /content/outputs/evaluation/alkaliphile_sweep.csv


In [ ]:
# alkaliphile: locked eval at the selected (matched) epoch -> verdict
ep=open("/content/outputs/evaluation/alkaliphile_epoch.txt").read().strip(); print("alkaliphile selected epoch:", ep)
!python /content/alkmpnn/evaluate.py --cohort alkaline --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ft_alk /content/outputs/alkaliphile_v1/model_weights/epoch_{ep}.pt \
    --ft_neu /content/outputs/alkaliphile_neu_v1/model_weights/epoch_{ep}.pt --n 8
import shutil
shutil.copy("/content/outputs/evaluation/verdict.txt","/content/outputs/evaluation/alkaliphile_verdict.txt")
shutil.copy("/content/outputs/evaluation/axis_eval_designs.csv","/content/outputs/evaluation/alkaliphile_axis_eval.csv")
print(open("/content/outputs/evaluation/alkaliphile_verdict.txt").read())


## Acidophile — fine-tune (expect designs BASIC (surface_net up))
Train the case model + the neutralophile control (FT_neu), select on val (max steer within the −3pp guardrail), eval on held-out neutralophile backbones. The `FT_neu` line in the verdict is the control.


In [ ]:
# acidophile: train FT_case + FT_neu (Noam warmup, checkpoint every 2 epochs)
!python /content/train.py --mpnn /content/ProteinMPNN --cohort acid --run acidophile_v1     --role case    --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 2 --schedule noam
!python /content/train.py --mpnn /content/ProteinMPNN --cohort acid --run acidophile_neu_v1 --role control --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 2 --schedule noam


In [ ]:
# acidophile: select checkpoint on VALIDATION (max steer within the recovery guardrail)
!python /content/alkmpnn/sweeps.py --cohort acid --mode checkpoint --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ckpt_dir /content/outputs/acidophile_v1/model_weights --split val --select maxsteer \
    --select_out /content/outputs/evaluation/acidophile_epoch.txt --save_csv /content/outputs/evaluation/acidophile_sweep.csv


In [ ]:
# acidophile: locked eval at the selected (matched) epoch -> verdict
ep=open("/content/outputs/evaluation/acidophile_epoch.txt").read().strip(); print("acidophile selected epoch:", ep)
!python /content/alkmpnn/evaluate.py --cohort acid --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ft_alk /content/outputs/acidophile_v1/model_weights/epoch_{ep}.pt \
    --ft_neu /content/outputs/acidophile_neu_v1/model_weights/epoch_{ep}.pt --n 8
import shutil
shutil.copy("/content/outputs/evaluation/verdict.txt","/content/outputs/evaluation/acidophile_verdict.txt")
shutil.copy("/content/outputs/evaluation/axis_eval_designs.csv","/content/outputs/evaluation/acidophile_axis_eval.csv")
print(open("/content/outputs/evaluation/acidophile_verdict.txt").read())


## Polar comparison — same recipe, opposite directions


In [ ]:
import re
def shift(label):
    t=open(f"/content/outputs/evaluation/{label}_verdict.txt").read()
    ft=float(re.search(r"surface_net\s+(-?\d+\.\d+)",t).group(1))
    neu=float(re.search(r"FT_neu surface_net shift ([+-]?\d+\.\d+)",t).group(1))
    return ft,neu
print("%-12s%20s%18s" % ("cohort","FT_case surface_net","FT_neu (control)"))
for lab in ["alkaliphile","acidophile"]:
    try:
        f,n=shift(lab); print("%-12s%20.3f%18.3f" % (lab,f,n))
    except Exception:
        print("%-12s  (run its eval cell first)" % lab)
print("\nExpected: alkaliphile FT_case NEGATIVE (acidic), acidophile FT_case POSITIVE (basic);")
print("each FT_neu control shifts the OPPOSITE way to its FT_case -> polar steering from one recipe.")
